###DAY 4 (11/01/26)– Delta Lake Introduction

### 🛠️ Tasks:


**1. USING PySpark - Creating Delta table from CSV File**


In [0]:
events_oct_df = spark.read.csv("/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv", header = True, inferSchema=True)

In [0]:
events_oct_df.write.format("delta").mode("overwrite").saveAsTable("workspace.ecommerce.events_oct_delta")

**2. USING SQL - Creating Delta table from another table**


In [0]:
spark.sql("""
          CREATE OR REPLACE TABLE events_delta
          USING DELTA
          AS SELECT * FROM workspace.ecommerce.events_oct_delta""")

**3. Test schema enforcement**

In [0]:
events_oct_df.printSchema()

In [0]:
### Define Strict Schema

from pyspark.sql.types import *

events_schema = StructType([
    StructField("event_time", TimestampType(), nullable=True),
    StructField("event_type", StringType(), nullable=True),
    StructField("product_id", IntegerType(), nullable=False),
    StructField("category_id", LongType(), nullable=True),
    StructField("category_code", StringType(), nullable=True),
    StructField("brand", StringType(), nullable=True),
    StructField("price", FloatType(), nullable=True),
    StructField("user_id", IntegerType(), nullable=True),
    StructField("user_session", StringType(), nullable=True)   
])

In [0]:
#Read raw data with schema

events_df = (
    spark.read
    .format("csv")
    .schema(events_schema) #enforced Schema here
    .option("header", True)
    .option("mode", "FAILFAST")  # Fail if bad records are found
    .load("/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv")
)

In [0]:
#Write data to Delta table
(
events_df.write
.format("delta")
.mode("append")
.saveAsTable("workspace.ecommerce.events_delta_enforced")
)

In [0]:
%sql
describe detail workspace.ecommerce.events_delta_enforced

In [0]:
%sql
describe workspace.ecommerce.events_delta_enforced

**4. Handling Duplicates Or Implementing SCD TYPE 1**

In [0]:
from delta.tables import DeltaTable
tgt_events = DeltaTable.forName(spark, "workspace.ecommerce.events_delta_enforced")
(
tgt_events.alias("tgt")
.merge(
    events_df.alias("src"),
    "tgt.product_id = src.product_id"
)
.whenMatchedUpdateAll()
.whenNotMatchedInsertAll()
.execute()
)